# The Idea is to do the tesitng runs

Initial run as always to th get the right working folder + One more for imports

In [2]:
import os
import sys

# 1. Change the working directory to the project root
%cd ..

# 2. Add the root directory to the Python path so imports work
sys.path.append(os.getcwd())

# 3. Verify we are in the right place
print(f"Current Working Directory: {os.getcwd()}")
# !ls # Optional: list files to confirm you see 'main.py' and 'pbi_utils'

/data/pavel.degterev/pbi
Current Working Directory: /data/pavel.degterev/pbi


In [3]:
import yaml
import torch
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch.nn.functional as F

from pbi_utils.data_manager import H5pyEmbeddingsManager, PerphectDataInput, EmbeddingsManager
from main import parse_config, Stats, make_dataset, create_embeddings_bacteria, create_embeddings_phages, logger

from pbi_utils.types import CACHED_EMBEDDINGS_OPTION # bool | Literal["auto"]

In [ ]:
# just in case, this is how df is saved: (may be for custom shapley embeddings)
#
# from sklearn.model_selection import train_test_split
# train, test = train_test_split(dataset, test_size=0.2, random_state=42, shuffle=True)
# torch.save(test, config.test_path)

In [ ]:
# just in case 2

# if technique == "PCA":
#     pca_bact = PCA(random_state=42, n_components=n_components_bact)
#     dataset["bacterium_embedding"] = list(torch.from_numpy(pca_bact.fit_transform(dataset["bacterium_embedding"].apply(lambda x: x.detach().cpu().numpy()).to_list())).float())  # type: ignore

#     pca_phag = PCA(random_state=42, n_components=n_components_phag)
#     dataset["phage_embedding"] = list(torch.from_numpy(pca_phag.fit_transform(dataset["phage_embedding"].apply(lambda x: x.detach().cpu().numpy()).to_list())).float())  # type: ignore

#     os.makedirs(config.output_dir, exist_ok=True)

#     with open(config.bacteria_pca_path, 'wb') as f: 
#         pickle.dump(pca_bact, f)
#     logger.info(f"Bacteria fitted PCA saved to: {config.bacteria_pca_path}")  

#     with open(config.phage_pca_path, 'wb') as f: 
#         pickle.dump(pca_phag, f)
#     logger.info(f"Phage fitted PCA saved to: {config.phage_pca_path}") 

## initialization

In [4]:
CONFIG_PATH = "model_configs/best_model_XAI.yaml"
config = parse_config(CONFIG_PATH)
DEVICE = config.device

[DEBUG] [NT2] Using InstaDeepAI default model weights


/home/pavel.degterev/miniforge3/envs/pbi/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


[DEBUG] [NT2] Max sequence length for Nucleotide Transformer: 12276
[DEBUG] [MegaDNA] Max sequence length for megaDNA: 131071
[DEBUG] [DNABERT2] Max sequence length for DNABERT2: 32768
[DEBUG] [NT2] Using InstaDeepAI default model weights
[DEBUG] [NT2] Max sequence length for Nucleotide Transformer: 12276
[DEBUG] [MegaDNA] Max sequence length for megaDNA: 131071
[DEBUG] [DNABERT2] Max sequence length for DNABERT2: 32768
[INFO] Configuration loaded from model_configs/best_model_XAI.yaml: Config(input_perphect=bacteria_df='data/perphect-data/all/bacteria_df.csv' phages_df='data/perphect-data/all/phages_df.csv' couples_df='data/perphect-data/all-private-oversampled/couples_df.csv', embeddings_dir=data/embeddings, num_gpu=1, gpu_id=0, training_config=TrainingConfig(do_train=True epochs=100 batch_size=256 learning_rate=0.001 weight_decay=0.0001 k_folds_cv=10 patience_early_stopping=1000 monitor_metric_early_stopping='f1' patience_reduce_lr=1000 monitor_metric_reduce_lr='f1' multiplying_fact

### Confighuration
With ability to run only those changes that actually needed

In [5]:
# I guess that here should be custom embeddings folder to not mess up OG embeddings
CUSTOM_EMBEDDINGS_FOLDER = "XAI/embeddings"
output_manager = H5pyEmbeddingsManager(CUSTOM_EMBEDDINGS_FOLDER)
#output_manager = H5pyEmbeddingsManager(config.embeddings_dir)

[INFO] Embeddings will be stored or read from XAI/embeddings


In [ ]:
CUSTOM_DATA_FOLDER = ''
config.input_perphect = CUSTOM_DATA_FOLDER

In [ ]:
CACHED_EMBEDDINGS_OPTION = False # bool | Literal["auto"]
config.compute_bacteria_embeddings = CACHED_EMBEDDINGS_OPTION
config.compute_phages_embeddings = CACHED_EMBEDDINGS_OPTION
# or ... but I guess it is better not to run this cell
config.compute_bacteria_embeddings = [False, False, False]
config.compute_phages_embeddings   = ["auto", "auto", False]

In [6]:
TEST_PAIR = [
    {"bacterium_id": 153, "phage_id": 2014},
]

### Data loading section
with 2 possible implementations (  Witihn config / `CUSTOM_RUN`)

In [7]:
if config.input_perphect is not None:
    bacteria_df, phages_df, couples_df = PerphectDataInput(
        input_paths=config.input_perphect  
    ).load()

[INFO] Perphect input files will be read from data/perphect-data/all/bacteria_df.csv, data/perphect-data/all/phages_df.csv and data/perphect-data/all-private-oversampled/couples_df.csv
[INFO] Reading csv files...


In [10]:
couples_df.head(
)

,id,phage_id,bacterium_id,interaction_type
0,5365,4968,1804,1
1,5366,4546,1804,1
2,5367,4911,1804,1
3,5368,4200,1804,1
4,5369,4942,1804,1


(optional / experemental) build custom df

In [8]:
pair_tuples = [(p["bacterium_id"], p["phage_id"]) for p in TEST_PAIR]

# 1. keep only requested pairs in couples_df
couples_df_filtered = couples_df[
    couples_df[["bacterium_id", "phage_id"]].apply(tuple, axis=1).isin(pair_tuples)
].copy()

# 2. collect IDs that are actually present after filtering
selected_bacteria_ids = couples_df_filtered["bacterium_id"].unique()
selected_phage_ids = couples_df_filtered["phage_id"].unique()

# 3. keep only matching rows in bacteria_df and phages_df
bacteria_df_filtered = bacteria_df[
    bacteria_df["bacterium_id"].isin(selected_bacteria_ids)
].copy()

phages_df_filtered = phages_df[
    phages_df["phage_id"].isin(selected_phage_ids)
].copy()

In [9]:
bacteria_df = bacteria_df_filtered.reset_index(drop=True)
phages_df = phages_df_filtered.reset_index(drop=True)
couples_df = couples_df_filtered.reset_index(drop=True)

In [16]:
print("bacteria_df:", bacteria_df.shape)
print("phages_df:", phages_df.shape)
print("couples_df:", couples_df.shape)

display(couples_df)

bacteria_df: (1, 3)
phages_df: (1, 3)
couples_df: (1, 4)


,id,phage_id,bacterium_id,interaction_type
0,788,2014,153,1


### Compiting the embeddings 

compute = config.models, compute CACHED_EMBEDDINGS_OPTION, df, output manager

In [10]:
create_embeddings_bacteria(
    config.bacteria_embedding_models,    # model in config
    config.compute_bacteria_embeddings,  # this is the CACHED_EMBEDDINGS_OPTION
    bacteria_df,                         # from the previous step
    output_manager,                      # ???               
)

create_embeddings_phages(
    config.phages_embedding_models,
    config.compute_phages_embeddings,
    phages_df,
    output_manager,
)

[INFO] Creating embeddings for 3 bacteria models...
[DEBUG] Creating bacteria embeddings for model NT2-TopBottomTruncateStrategy-250M-ov0...


100%|██████████| 1/1 [00:00<00:00, 114.52it/s]

[DEBUG] Saving 1 embeddings for model NT2-TopBottomTruncateStrategy-250M-ov0 to XAI/embeddings. Shape: torch.Size([1536])


Saving embeddings:   0%|          | 0/1 [00:00<?, ?it/s]

[DEBUG] Creating bacteria embeddings for model MegaDNA-BottomTruncateStrategy-concat-ov0...


...

Saving embeddings:   0%|          | 0/1 [00:00<?, ?it/s]

[DEBUG] Creating phage embeddings for model DNABERT2-TKPert-concat-J16-g20-ov0-maxlen32768...


  0%|          | 0/1 [00:00<?, ?it/s]/home/pavel.degterev/.cache/huggingface/modules/transformers_modules/bert_layers.py:433: UserWarning: Increasing alibi size from 512 to 7001
  warnings.warn(
100%|██████████| 1/1 [00:11<00:00, 11.49s/it]

[DEBUG] Saving 1 embeddings for model DNABERT2-TKPert-concat-J16-g20-ov0-maxlen32768 to XAI/embeddings. Shape: torch.Size([1, 12288])


Saving embeddings:   0%|          | 0/1 [00:00<?, ?it/s]

### Creating or loading the dataset aka meta embeddings

calculating the embeddings

In [11]:
# compute option
bacteria_model_names = [x.name() for x in config.bacteria_embedding_models]
phages_model_names = [x.name() for x in config.phages_embedding_models]

emb_test_data = make_dataset(
    couples_df,                       # got it in pre... previous step ?
    bacteria_model_names,             # here
    phages_model_names,               # here
    output_manager,                   # ???
    DEVICE                            # from initialization phase
    )

[INFO] Creating dataset (loading embeddings)...
[DEBUG] Loading 1 embeddings for model NT2-TopBottomTruncateStrategy-250M-ov0 from XAI/embeddings


Loading embeddings:   0%|          | 0/1 [00:00<?, ?it/s]

[DEBUG] Loading 1 embeddings for model MegaDNA-BottomTruncateStrategy-concat-ov0 from XAI/embeddings


Loading embeddings:   0%|          | 0/1 [00:00<?, ?it/s]

[DEBUG] Loading 1 embeddings for model DNABERT2-TopBottomTruncateStrategy-ov0-maxlen32768 from XAI/embeddings


...

[DEBUG] Loading 1 embeddings for model MegaDNA-MaxStrategy-concat-ov0 from XAI/embeddings


Loading embeddings:   0%|          | 0/1 [00:00<?, ?it/s]

[DEBUG] Loading 1 embeddings for model DNABERT2-TKPert-concat-J16-g20-ov0-maxlen32768 from XAI/embeddings


Loading embeddings:   0%|          | 0/1 [00:00<?, ?it/s]

[DEBUG] Final embedding size (bacteria): 4036
[DEBUG] Final embedding size (phages): 14788


alternatively we can load previously computed dataset (embeddings)

In [8]:
# load option
emb_test_data = torch.load(config.test_path, map_location=DEVICE)

In [9]:
emb_test_data.head() # embeddings from the test part in the training stage

,id,phage_id,bacterium_id,interaction_type,bacterium_embedding,phage_embedding
5397,7899,5286,5180,1,"[tensor(-4.4061, device='cuda:0'), tensor(-0.5...","[tensor(0.5544, device='cuda:0'), tensor(-1.14..."
3077,9255,5206,5244,1,"[tensor(-3.5581, device='cuda:0'), tensor(0.69...","[tensor(-6.6408, device='cuda:0'), tensor(-1.0..."
6050,7810,5283,5178,1,"[tensor(-4.5484, device='cuda:0'), tensor(2.29...","[tensor(-1.2678, device='cuda:0'), tensor(-2.0..."
6502,1127,2318,1869,1,"[tensor(3.3656, device='cuda:0'), tensor(-2.83...","[tensor(6.8565, device='cuda:0'), tensor(-0.74..."
3601,5366,4546,1804,1,"[tensor(-9.7324, device='cuda:0'), tensor(1.74...","[tensor(-6.0762, device='cuda:0'), tensor(0.92..."


### Loading from the config and appling PCA

In [12]:
with open(config.bacteria_pca_path, 'rb') as f:
    pca_bact = pickle.load(f)
emb_test_data["bacterium_embedding"] = list(torch.from_numpy(pca_bact.transform(emb_test_data["bacterium_embedding"].apply(lambda x: x.detach().cpu().numpy()).to_list())).float())  # type: ignore

with open(config.phage_pca_path, 'rb') as f:
    pca_phag = pickle.load(f)
emb_test_data["phage_embedding"] = list(torch.from_numpy(pca_phag.transform(emb_test_data["phage_embedding"].apply(lambda x: x.detach().cpu().numpy()).to_list())).float())  # type: ignore


### Loading model

In [13]:
bacterium_embed_size = len(emb_test_data["bacterium_embedding"].iloc[0]) # always 500 ???
phage_embed_size = len(emb_test_data["phage_embedding"].iloc[0])
model = config.classifier(bacterium_embed_size, phage_embed_size, **config.classifier_params)

model.load_state_dict(torch.load(config.model_path, map_location=DEVICE))

/home/pavel.degterev/miniforge3/envs/pbi/lib/python3.10/site-packages/torch/nn/init.py:412: UserWarning: Initializing zero-element tensors is a no-op
  warnings.warn("Initializing zero-element tensors is a no-op")


<All keys matched successfully>

### Model prediction

In [17]:
DEVICE

'cuda:0'

In [18]:
# Do 1 prediction example
example = emb_test_data.iloc[0]
bacterium_embedding = example["bacterium_embedding"].clone().detach().unsqueeze(0).to(DEVICE)
phage_embedding = example["phage_embedding"].clone().detach().unsqueeze(0).to(DEVICE)
model.eval()
model.to(DEVICE)
with torch.no_grad():
    #output = model(bacterium_embedding, phage_embedding)
    #probs = torch.sigmoid(output).squeeze().cpu().numpy()

    output = model(bacterium_embedding, phage_embedding)              # [1, 2]
    probs = torch.softmax(output, dim=1)     # [1, 2], корректные вероятности
    probs_np = probs.squeeze().cpu().numpy()
    pred_label = probs_np.argmax()
    pred_prob = probs_np[pred_label]

    print(f"Predicted interaction probability for bacterium {example['bacterium_id']} and phage {example['phage_id']}: Label {np.argmax(probs_np)} with probability {np.max(probs_np):.6f}. True label: {example['interaction_type']}")

Predicted interaction probability for bacterium 153 and phage 2014: Label 1 with probability 0.999852. True label: 1


In [19]:
# Pick the row for bacterium 153 & phage 2011
row = emb_test_data[
    (emb_test_data["bacterium_id"] == 153) &
    (emb_test_data["phage_id"] == 2014)
].iloc[0]

# Prepare embeddings (batch size = 1)
bact = row["bacterium_embedding"].clone().detach().unsqueeze(0).to(DEVICE)
phage = row["phage_embedding"].clone().detach().unsqueeze(0).to(DEVICE)

# Predict
model.eval()
with torch.no_grad():
    logits = model(bact, phage)
    probs = torch.softmax(logits, dim=1)
    probs_np = probs.squeeze().cpu().numpy()

pred_label = probs_np.argmax()
pred_prob = probs_np[pred_label]

# pred_label = int(np.argmax(probs))
# pred_prob = float(np.max(probs))

print(
    f"Predicted interaction probability for bacterium {row['bacterium_id']} "
    f"and phage {row['phage_id']}: Label {pred_label} with probability {pred_prob:.6f}. "
    f"True label: {row['interaction_type']}"
)

Predicted interaction probability for bacterium 153 and phage 2014: Label 1 with probability 0.999852. True label: 1
